# CardioTwin - Model Training (Estudiante B - Día 1)
Este notebook aisla el trabajo de Machine Learning. Se encarga de:
1. Cargar el dataset de Framingham.
2. Limpiar nulos.
3. Aplicar SMOTE para manejar el desbalance (CHD ~15%).
4. Entrenar XGBoost.
5. Evaluar con AUC-ROC.
6. Serializar el modelo con joblib.
7. Integrar SHAP y validar.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import shap
import joblib

# 1. Cargar Framingham
df = pd.read_csv('../../data/framingham.csv')
print("Shape original:", df.shape)
df.head()

In [ ]:
# 2. Limpiar nulos
df = df.dropna()
print("Shape sin nulos:", df.shape)

# Separar features (X) y target (y)
X = df.drop(columns=['TenYearCHD'])
y = df['TenYearCHD']

# Verificar prevalencia
print("\nPrevalencia CHD:")
print(y.value_counts(normalize=True))

In [ ]:
# Split de datos (antes de SMOTE para evitar data leakage en testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 3. Aplicar SMOTE
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print("Shape tras SMOTE:", X_train_resampled.shape)
print("Prevalencia en train resampled:")
print(y_train_resampled.value_counts(normalize=True))

In [ ]:
# 4. Entrenar XGBoost
model = xgb.XGBClassifier(
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42
)
model.fit(X_train_resampled, y_train_resampled)


In [ ]:
# 5. Evaluar con AUC-ROC
y_pred_proba = model.predict_proba(X_test)[:, 1]
y_pred = model.predict(X_test)

auc = roc_auc_score(y_test, y_pred_proba)
print(f"AUC-ROC: {auc:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
# 6. Serializar el modelo con joblib
joblib.dump(model, '../xgboost_chd_model.joblib')
print("Modelo serializado en '../xgboost_chd_model.joblib'")

In [ ]:
# 7. Integrar shap.TreeExplainer y validar
explainer = shap.TreeExplainer(model)

# Tomar un registro de ejemplo
sample = X_test.iloc[[0]]
shap_values = explainer.shap_values(sample)

print("Valores SHAP para el registro de ejemplo:")
print(shap_values)

# Mostrar importancia base de features (opcional en Jupyter, descomentar si usas la UI)
# shap.summary_plot(explainer.shap_values(X_test), X_test, plot_type="bar")